# Lesson 6 · Putting it together: retrieval-augmented detection

<a href="https://colab.research.google.com/github/Mary-Vadi/rag/blob/main/lessons/06_retrieval_augmented_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**In this lesson you will**

- combine the classifier (lesson 5) and retrieval (lesson 4) with a single number, **α** (alpha)
- understand the **decision window**: when retrieval can and can't change the answer
- tune α properly on the validation set and report the test result once
- compare two kinds of embeddings for retrieval: general-purpose and detector-trained

**Runtime:** a **GPU is needed** (Runtime → Change runtime type → T4 GPU).

## 1. The blend

Both halves of our detector output `P(machine)`. We combine them with a weighted average:

$$P_{\text{final}} = \alpha \cdot P_{\text{classifier}} + (1 - \alpha) \cdot P_{\text{retrieval}}$$

- α = 1: only the classifier counts
- α = 0: only retrieval counts
- α = 0.7: the setting used in the original experiment (70% classifier, 30% retrieval)

As always, `P_final ≥ 0.5` means "machine".

The idea: the classifier brings what it learned from RAID; retrieval brings labelled examples **from the new domain** (MAGE), without any retraining.

## 2. Setup

In [ ]:
# Setup: run this cell first.
# In Colab it downloads the course code and installs the extra libraries.
# On your own computer (from the lessons/ folder) it just makes the code importable.
import os, sys

if "google.colab" in sys.modules:
    if not os.path.exists("/content/rag"):
        !git clone -q https://github.com/Mary-Vadi/rag.git /content/rag
        !pip install -q -r /content/rag/requirements.txt
    %cd /content/rag
elif os.path.basename(os.getcwd()) == "lessons":
    %cd ..
sys.path.insert(0, os.getcwd())

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.model_selection import train_test_split

from ragdetect.augment import blend, decision_window, tune_alpha
from ragdetect.classifier import predict_proba, train_classifier
from ragdetect.data import load_mage, load_raid
from ragdetect.embeddings import Embedder
from ragdetect.evaluation import compare, evaluate, plot_confusion
from ragdetect.retrieval import RetrievalDetector

print("GPU available:", torch.cuda.is_available())

## 3. The classifier from lesson 5

Colab gives every notebook its own fresh machine, so the model trained in lesson 5 usually isn't here. If it's missing, the next cell trains it again with exactly the same data and settings (a few minutes on a GPU).

In [ ]:
model_dir = "models/classifier"

if not os.path.exists(os.path.join(model_dir, "config.json")):
    raid = load_raid(n=4000)
    raid_train, raid_rest = train_test_split(raid, test_size=0.2, stratify=raid["label"], random_state=42)
    raid_val, _ = train_test_split(raid_rest, test_size=0.5, stratify=raid_rest["label"], random_state=42)
    train_classifier(raid_train, raid_val, output_dir=model_dir, epochs=1, max_length=256)
print("classifier ready in", model_dir)

## 4. The MAGE data: pool, validation, test

The same three sets as in lesson 4, each with its own job.

In [ ]:
pool = load_mage("train", n=5000)
validation = load_mage("validation", n=1000)
test = load_mage("test", n=2000)

## 5. The classifier's opinion

In [ ]:
classifier_proba = {
    "validation": predict_proba(model_dir, validation["text"]),
    "test": predict_proba(model_dir, test["text"]),
}

## 6. Retrieval's opinion, with two kinds of embeddings

In lesson 2 we saw that a general-purpose model like MiniLM groups texts mainly by topic. Our fine-tuned ModernBERT has spent its training learning to separate human from machine text, so its embeddings might group texts differently. That is what the original experiment used. Let's try both.

We use a plain vote over `k = 7` neighbours, as in the original experiment. (You can swap in the best `k` you found in lesson 4.)

In [ ]:
embedding_models = {
    "MiniLM": "sentence-transformers/all-MiniLM-L6-v2",
    "fine-tuned ModernBERT": model_dir,
}

retrieval_proba = {}
for name, path in embedding_models.items():
    print(f"Embedding with {name} ...")
    embedder = Embedder(path)
    detector = RetrievalDetector(k=7, temperature=None).fit(embedder.encode(pool["text"], show_progress=True), pool["label"])
    retrieval_proba[name] = {
        split: detector.predict_proba(embedder.encode(df["text"], show_progress=True))
        for split, df in [("validation", validation), ("test", test)]
    }
    del embedder
    torch.cuda.empty_cache()

When the fine-tuned model is loaded as an embedder, a "LOAD REPORT" lists the `classifier` and `head` weights as **UNEXPECTED**. That's fine: for embeddings we only need the main body of the model, so its classification head is simply left out.

## 7. The original recipe: α = 0.7

Let's reproduce the original set-up on our smaller sample and compare it with each half on its own.

In [ ]:
y_test = test["label"]
results = {"classifier only": evaluate(y_test, classifier_proba["test"])}
for name in embedding_models:
    results[f"retrieval only ({name})"] = evaluate(y_test, retrieval_proba[name]["test"])
for name in embedding_models:
    results[f"blend α=0.7 ({name})"] = evaluate(y_test, blend(classifier_proba["test"], retrieval_proba[name]["test"], alpha=0.7))
compare(results)

## 8. The decision window: why α = 0.7 can barely help

Retrieval's vote is always between 0 and 1. So with α = 0.7, the final score is somewhere between:

- `0.7 × P_classifier + 0.3 × 0` (every neighbour says human), and
- `0.7 × P_classifier + 0.3 × 1` (every neighbour says machine).

Work it through. If the classifier says `P = 0.9`, the lowest possible final score is `0.7 × 0.9 = 0.63`, still above 0.5. **Even if all seven neighbours say "human", the answer stays "machine".**

In general, retrieval can only change the decision when the classifier's score lies inside a **decision window**. `decision_window` in `ragdetect/augment.py` works it out:

In [ ]:
for alpha in [0.9, 0.7, 0.6, 0.5]:
    low, high = decision_window(alpha)
    print(f"α = {alpha}: retrieval can change the answer only if the classifier's P(machine) is between {low:.2f} and {high:.2f}")

Now the key question: **how many of our test texts actually fall inside the window for α = 0.7?**

In [ ]:
low, high = decision_window(0.7)
p = classifier_proba["test"]
inside = (p > low) & (p < high)
print(f"{inside.mean():.1%} of test texts are inside the window ({low:.2f} to {high:.2f})")

plt.figure(figsize=(7, 3.5))
plt.hist(p, bins=40, range=(0, 1), color="grey")
plt.axvspan(low, high, color="tab:green", alpha=0.25, label="retrieval can change the answer")
plt.xlabel("classifier's P(machine) on MAGE test")
plt.ylabel("number of texts")
plt.legend()
plt.show()

If the classifier is confident about almost everything, as we saw in lesson 5, very few texts land in the window, and retrieval hardly ever gets a say, however good its vote is. In the original experiment retrieval improved accuracy by just **0.4 points**, and this is a likely explanation.

The fix is not to guess a better α, but to **learn it from data**.

## 9. Tuning α on the validation set

`tune_alpha` tries α = 0, 0.05, 0.10, …, 1 on the **validation** set and returns the best one.

In [ ]:
chosen_alpha = {}
plt.figure(figsize=(7, 3.5))
for name in embedding_models:
    best, table = tune_alpha(validation["label"], classifier_proba["validation"], retrieval_proba[name]["validation"])
    chosen_alpha[name] = best
    plt.plot(table["alpha"], table["accuracy"], marker="o", markersize=3, label=f"{name} (best α = {best:.2f})")
plt.axvline(0.7, color="black", linestyle="--", linewidth=1, label="original α = 0.7")
plt.xlabel("α (weight on the classifier)")
plt.ylabel("validation accuracy")
plt.legend()
plt.show()
chosen_alpha

Reading the curve: the left end (α = 0) is retrieval on its own and the right end (α = 1) is the classifier on its own. If the curve peaks somewhere in the middle, the two really do complement each other. If it keeps rising towards one end, that side is simply better on this data.

## 10. The final test result

Now we apply each tuned α to the test set, **once**.

In [ ]:
for name in embedding_models:
    alpha = chosen_alpha[name]
    results[f"blend α={alpha:.2f}, tuned ({name})"] = evaluate(
        y_test, blend(classifier_proba["test"], retrieval_proba[name]["test"], alpha=alpha)
    )
compare(results)

In [ ]:
best_name = max(embedding_models, key=lambda name: results[f"blend α={chosen_alpha[name]:.2f}, tuned ({name})"]["accuracy"])
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
plot_confusion(y_test, classifier_proba["test"], title="Classifier only", ax=axes[0])
plot_confusion(
    y_test, blend(classifier_proba["test"], retrieval_proba[best_name]["test"], alpha=chosen_alpha[best_name]),
    title=f"Tuned blend ({best_name})", ax=axes[1],
)
plt.tight_layout()
plt.show()

## 11. What have we learned?

Look at your final table and think about these questions. There's no single right answer; your numbers depend on the sample and the run.

1. **Did retrieval help at all?** Compare "classifier only" with the tuned blends.
2. **Which embeddings retrieved better neighbours?** Compare the two "retrieval only" rows. Was the general-purpose model or the detector-trained one better at finding same-label neighbours?
3. **Did tuning α matter?** Compare "α=0.7" with "tuned" for each embedding model, and look back at the decision window.

<details>
<summary>What the original experiment found</summary>

With α = 0.7 and embeddings from the fine-tuned ModernBERT, retrieval raised MAGE accuracy from 53.8% to 54.2%. The most likely explanation is that the classifier was confidently wrong on most human MAGE texts, leaving them outside the decision window where retrieval couldn't reach them. Tuning α on validation data and trying different embeddings are the two most direct ways to give retrieval a real chance, which is what you've just done.
</details>

### An honest caveat

Retrieval used **labelled** MAGE texts (from the train split). So this is not a "zero-shot" result: it answers the question *"if I have some labelled examples from a new domain, can I use them without retraining?"* That's a realistic and useful situation, but it's a different claim from "the detector works on any text". Always state which one you're making.

### Back to RAG

We used the **retrieval** half of RAG: embeddings, a vector index, nearest neighbours. A natural next step is to add the **generation** half too: retrieve the most similar labelled texts, show them to an LLM as examples ("these were written by people, these by AI"), and ask it to judge the new text. That would be retrieval-augmented generation applied to detection.

## Check your understanding

<details>
<summary>1. With α = 0.8, can retrieval change the decision for a text the classifier scores at 0.7?</summary>

The window for α = 0.8 is 0.375 to 0.625. A score of 0.7 is outside it: the final score is at least 0.8 × 0.7 = 0.56, so the answer stays "machine" whatever the neighbours say.
</details>

<details>
<summary>2. Why tune α on the validation set and not the test set?</summary>

The test set has to stay untouched until the end so that its score is an honest estimate on new data. Tuning on it would quietly overfit our choice of α to those exact texts.
</details>

<details>
<summary>3. Why might embeddings from the fine-tuned classifier retrieve better (or worse) neighbours than MiniLM?</summary>

Better: fine-tuning pushed its representations to separate human from machine text, which is exactly what we want neighbours to share. Worse: it was fine-tuned only on RAID abstracts, so on MAGE its embedding space may be organised around clues that don't transfer, the same domain-shift problem as the classifier itself.
</details>

## Project ideas

1. **Bigger pool:** retrieval gets better the more relevant labelled examples it can search. Try 20,000 or 50,000 pool texts.
2. **Tune everything together:** search over `k`, temperature and α at once on the validation set.
3. **Train on all of RAID's domains** and check whether the classifier's MAGE score (and retrieval's added value) changes.
4. **Full RAG:** give the retrieved examples to an LLM and ask it to decide, as described above.

## Course summary

| Lesson | Key idea |
|---|---|
| 1 | Detection is binary classification; retrieval-augmented detection blends a classifier with a vote from similar labelled texts. |
| 2 | Embeddings turn text into vectors; cosine similarity compares them. |
| 3 | Nearest-neighbour search finds similar texts; FAISS makes it fast. |
| 4 | Neighbours can vote; measure with several metrics; tune on validation, test once. |
| 5 | A fine-tuned classifier can be excellent in-domain and poor out-of-domain. |
| 6 | Retrieval brings in knowledge from the new domain, but only if the blend gives it room to act. |

The full-size original experiment, with its results, is in the [`experiments/`](../experiments) folder.